In [21]:
import pandas as pd, numpy as np, os, json, re
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import  warnings
from sklearn.model_selection import KFold
from scipy.stats import chi2_contingency, norm, mannwhitneyu
warnings.filterwarnings("ignore")
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)


EDU_MAP = {"Illiterate": 0, "Literacy": 1, "Pre-Primary": 2, "Primary": 3, "PSC": 3, "JSC": 4,
           "Secondary": 4, "SSC": 5, "Higher-Secondary": 6, "HSC": 6, "Graduation": 7,
           "Honors": 7, "Degree": 7, "Post-Graduation": 8, "Masters": 8, "PHD": 9}
BIG = 300000

CSV_PATH = os.environ.get("LOAN_CSV", "dataset/SingleCycleLoaneeMember.csv")
R = os.path.join('./', "results")

SA = json.load(open(os.path.join(R, "selection_trackA.json")))["selected"]
BEST = json.load(open(os.path.join(R, "best_params.json")))["A"]


def load():
    df = pd.read_csv(CSV_PATH)
    df["_adm"] = pd.to_datetime(df["admission_date"])
    return df.sort_values(["_adm", "member_id"]).reset_index(drop=True)

PARAMS = dict(learning_rate=0.08, num_leaves=31, min_child_samples=50, subsample=0.8, subsample_freq=1,
              colsample_bytree=0.8, reg_lambda=5, cat_smooth=30, min_data_per_group=50, cat_l2=10,
              max_cat_to_onehot=4, n_estimators=2500, verbose=-1, n_jobs=1)



def clean_id(s: pd.Series) -> pd.Series:
    return s.astype("string").str.strip().str.lstrip("0").fillna("NA").astype(str)

def clean_text_purpose(s: pd.Series) -> pd.Series:
    return s.fillna("NA").astype(str).str.replace(r"\s*\(.*$", "", regex=True).str.strip()

def altocc_group(s):
    if pd.isna(s): return "none"
    s = str(s).strip().lower()
    if re.search(r"agri|krish|krisi|farm|kisi|dairy|fish|cultiv", s): return "agri"
    if re.search(r"bus|shop|dokan|trad", s): return "business"
    if re.search(r"job|serv", s): return "job"
    if re.search(r"probas|abroad|foreign|immig", s): return "abroad"
    if re.search(r"house|home", s): return "housewife"
    if re.search(r"driv|auto|rickshaw|van", s): return "driver"
    return "OTHERS"

def purpose_group(s: str) -> str:
    s = s.lower()
    if re.search(r"immigrant|abroad", s): return "Migration"
    if re.search(r"dairy|goru|cattle fatt|cow|goat|poultry|duck|sheep|livestock|animal|motataja|rearing|pigeon|farm(ing)?$", s): return "Livestock"
    if re.search(r"fish|shrimp|pond", s): return "Fishery"
    if re.search(r"paddy|vegetable cult|potato|jute|onion|betel|corn|maize|cultiv|wheat|mustard|chili|garlic|banana|mango|tobacco|crop|nursery|agricultur|seed", s): return "Crop"
    if re.search(r"house|home|building|repair|toilet|latrine|tube|solar|sanitation|water", s): return "Housing"
    if re.search(r"auto|vehicle|rickshaw|van|transport|truck|cng|bus$|tempo|motor|bike|boat|driver|easy bike|battery", s): return "Transport"
    if re.search(r"business|shop|store|trade|dokan|stall|tea|grocery|cloth|garment|hardware|restaurant|hotel|medicine|pharmacy|distributor|mobile|electric|jewel|crockery|sanitary|brick|saw|mill|tailor|cosmetic|furniture|fertilizer|feed|wholesale|retail|bakery|food|fast|sweet|salon|parlor|barber|gas|plastic|glass", s): return "Trade"
    return "OTHER"

In [22]:
# """ from total loan """
# 1431450 Jagaron ( max limit 500k)
# 927558 bellow 100k
# 3457   greater than equal 300k
# 823 greater than 300k
# 285773 agrosor
# 239983 greater than equal 300k
# 143316 greater than 300k
# 26  bellow 100k

In [24]:
df=load()
df.head()

,member_id,name,dateofbirth,gender,current_addr_district,current_addr_thana,maritial_status,admission_date,education,occupation,alternative_occupation,total_family_member,total_earning_member,total_monthly_income,yearly_total_income,yearly_total_expenses,yearly_total_savings,home_owner_type,year_of_rent,farm_land_amt,loan_id,proposer_oldid,loan_cycle_no,loan_Component,loan_use_sources,applied_loan_amount,loan_amount,installmenttype,installmentyear,totalInstallmentcount,_adm
0,405785,Kajal akter,1972-02-05,Female,Feni,Parshuram,Married,2004-06-20,Secondary,Housewife,Agriculture,5,3,100000.00,1200000.00,600000.00,600000.00,Own,0.00,60,1849565,0115690915,1,JAGORON,Poultry Farming (হাঁস-মুরগি পালন),50000.00,50000.00,Mounthly,1.00,12,2004-06-20
1,176729,Mst.Nazma Begum,1982-08-02,Female,Rajshahi,Paba,Married,2006-02-05,PSC,Housewife,NaN,3,1,31000.00,372000.00,50000.00,70000.00,Own,0.00,33,1827694,0163451221,1,SUFOLON,Goat Rearing (ছাগল পালন),50000.00,50000.00,Weekly,1.00,46,2006-02-05
2,350207,Senowara Begum,1980-04-12,Female,Chattogram,Rangunia,Married,2006-05-21,Primary,Housewife,NaN,5,2,25000.00,300000.00,160000.00,140000.00,Own,0.00,20,1633560,0182190323,1,SAHOS,Poultry Farming (হাঁস-মুরগি পালন),20000.00,20000.00,Mounthly,1.00,12,2006-05-21
3,236626,Sandha Rani Das,1977-12-10,Female,Barishal,Uzirpur,Married,2006-11-20,Secondary,Housewife,Job,3,2,80000.00,960000.00,450000.00,510000.00,Own,0.00,40,1693230,0085940211,1,JAGORON,Betel Leaf Cultivation (পান চাষ),90000.00,90000.00,Weekly,1.00,46,2006-11-20
4,56426,Amina,1970-01-11,Female,Rangpur,Gangachara,Married,2008-02-20,PSC,Agriculture,NaN,4,2,20000.00,240000.00,120000.00,120000.00,Own,0.00,60,1689673,0184340523,1,JAGORON,Paddy Cultivation (ধান চাষ),40000.00,40000.00,Weekly,1.00,46,2008-02-20


In [25]:
print("Agrosor median",df[df['loan_Component'] == 'AGROSOR']['loan_amount'].median())
print("Jagoron median",df[df['loan_Component'] == 'JAGORON']['loan_amount'].median())

Agrosor median 300000.0
Jagoron median 60000.0


In [26]:
print("Agrosor min",df[df['loan_Component'] == 'AGROSOR']['loan_amount'].min())
print("Jagoron min",df[df['loan_Component'] == 'JAGORON']['loan_amount'].min())

Agrosor min 300000.0
Jagoron min 5000.0


In [27]:
print("Agrosor max",df[df['loan_Component'] == 'AGROSOR']['loan_amount'].max())
print("Jagoron max",df[df['loan_Component'] == 'JAGORON']['loan_amount'].max())

Agrosor max 5000000.0
Jagoron max 295000.0


In [28]:
print("Agrosor mean",df[df['loan_Component'] == 'AGROSOR']['loan_amount'].mean().round(2))
print("Jagoron mean",df[df['loan_Component'] == 'JAGORON']['loan_amount'].mean().round(2))

Agrosor mean 405052.28
Jagoron mean 76550.55


In [29]:
df.loc[
    df['loan_Component'] == 'AGROSOR'
].describe().drop(columns=['member_id', 'loan_id', 'loan_cycle_no', 'applied_loan_amount'])

,total_family_member,total_earning_member,total_monthly_income,yearly_total_income,yearly_total_expenses,yearly_total_savings,year_of_rent,farm_land_amt,loan_amount,installmentyear,totalInstallmentcount,_adm
count,10405.00,10405.00,10405.00,10405.00,10405.00,10405.00,10400.00,10405.00,10405.00,10405.00,10405.00,10405
mean,4.35,1.90,181481.50,2177777.95,767513.98,1400262.82,12.75,6168.70,405052.28,1.02,19.77,2025-02-18 10:13:38.606439168
min,1.00,0.00,10000.00,120000.00,2000.00,0.00,0.00,0.00,300000.00,1.00,12.00,2009-10-20 00:00:00
25%,4.00,1.00,80000.00,960000.00,300000.00,510000.00,0.00,0.00,300000.00,1.00,12.00,2024-12-23 00:00:00
50%,4.00,2.00,105000.00,1260000.00,480000.00,760000.00,0.00,30.00,300000.00,1.00,12.00,2025-03-03 00:00:00
75%,5.00,2.00,170000.00,2040000.00,840000.00,1200000.00,0.00,99.00,500000.00,1.00,18.00,2025-05-18 00:00:00
max,21.00,8.00,18000000.00,216000000.00,50000000.00,215820000.00,25000.00,60000000.00,5000000.00,2.00,67.00,2026-07-23 00:00:00
std,1.27,0.86,435116.15,5221393.84,1318966.82,4782447.17,497.68,588568.98,208983.93,0.11,14.00,NaN


In [30]:
df.loc[
    df['loan_Component'] == 'JAGORON'
].describe().drop(columns=['member_id', 'loan_id', 'loan_cycle_no', 'applied_loan_amount'])

,total_family_member,total_earning_member,total_monthly_income,yearly_total_income,yearly_total_expenses,yearly_total_savings,year_of_rent,farm_land_amt,loan_amount,installmentyear,totalInstallmentcount,_adm
count,90278.00,90278.00,90278.00,90278.00,90278.00,90278.00,90269.00,90278.00,90278.00,90278.00,90278.00,90278
mean,3.97,1.69,68767.20,825206.37,280260.31,485039.75,2.10,24722.45,76550.55,1.00,35.83,2025-02-16 12:59:14.460666112
min,1.00,0.00,250.00,3000.00,150.00,0.00,0.00,0.00,5000.00,1.00,12.00,2004-06-20 00:00:00
25%,3.00,1.00,35000.00,420000.00,160000.00,180000.00,0.00,0.00,50000.00,1.00,12.00,2024-12-19 00:00:00
50%,4.00,2.00,45000.00,540000.00,240000.00,270000.00,0.00,10.00,60000.00,1.00,46.00,2025-02-27 00:00:00
75%,5.00,2.00,60000.00,720000.00,350000.00,420000.00,0.00,45.00,100000.00,1.00,46.00,2025-05-18 00:00:00
max,23.00,10.00,500001554.00,6000018648.00,96000000.00,935500000.00,23000.00,2147483647.00,295000.00,2.00,90.00,2026-07-23 00:00:00
std,1.05,0.71,1694521.25,20334254.98,499097.75,3841397.03,137.16,7147654.95,47886.06,0.01,15.57,NaN


In [ ]:
overdue --

In [31]:
data = load()
a=data[data['loan_amount'] >= 300000].copy()
a['total_monthly_income'].median()

np.float64(105000.0)

In [32]:
a['total_monthly_income'].mean()

np.float64(181426.4057888763)

In [33]:
df.loc[df['loan_Component'] == 'AGROSOR-RAISE']['loan_amount'].min()

np.float64(52000.0)

In [34]:
OUT = os.path.join('./', "results")
os.makedirs(OUT, exist_ok=True)

df["agrosor"] = df["loan_Component"].isin(["AGROSOR", "AGROSOR-MFCE"])
df['jagoron'] = df['loan_Component'].isin(['JAGORON'])
adm = pd.to_datetime(df["admission_date"])
df["age"] = (adm - pd.to_datetime(df["dateofbirth"])).dt.days / 365.25
df["adm_year"] = adm.dt.year + adm.dt.month / 12
df["monthly_income"] = df["total_monthly_income"]
df["yearly_expenses"] = df["yearly_total_expenses"]
df["yearly_savings"] = df["yearly_total_savings"]
df["land"] = df["farm_land_amt"].where(df["farm_land_amt"] < 1e5)               # drop the int32 sentinel
df["family"] = df["total_family_member"]
df["earners"] = df["total_earning_member"]
df["expense_ratio"] = (df["yearly_total_expenses"] / df["yearly_total_income"].replace(0, np.nan)).clip(0, 3)
df["income_per_member"] = df["total_monthly_income"] / df["total_family_member"].replace(0, np.nan)
df["income_per_earner"] = df["total_monthly_income"] / df["total_earning_member"].replace(0, np.nan)
df["earner_ratio"] = df["total_earning_member"] / df["total_family_member"].replace(0, np.nan)
df["rent_years"] = df["year_of_rent"]
pur = clean_text_purpose(df["loan_use_sources"])
df["purpose"] = pur
df["purpose_family"] = pur.map(purpose_group)
df["alt_occupation_group"] = df["alternative_occupation"].map(altocc_group)
df["home_owner"] = df["home_owner_type"].fillna("(missing)")
df["district"] = df["current_addr_district"].astype(str).str.strip()
df["thana"] = df["current_addr_thana"].astype(str).str.strip()
df["education"] = df["education"].fillna("(missing)")

A, O = df[df.agrosor], df[df.jagoron]
print(f"AGROSOR (AGROSOR + AGROSOR-MFCE ): {len(A):,} loans ({len(A) / len(df):.1%})   JAGORON: {len(O):,}")
print("excluded from AGROSOR group on purpose: AGROSOR-RAISE (578 loans, amounts 52k-700k) stays in JAGORON\n")
print("AGROSOR loan amounts:", A.loan_amount.value_counts(normalize=True).head(5).round(3).to_dict())

AGROSOR (AGROSOR + AGROSOR-MFCE ): 10,480 loans (8.5%)   JAGORON: 90,278
excluded from AGROSOR group on purpose: AGROSOR-RAISE (578 loans, amounts 52k-700k) stays in JAGORON

AGROSOR loan amounts: {300000.0: 0.619, 500000.0: 0.201, 400000.0: 0.084, 800000.0: 0.051, 1000000.0: 0.014}


In [36]:
#1. numeric features
num = ["monthly_income", "yearly_expenses", "yearly_savings", "land", "family", "earners", "age", "rent_years",
       "expense_ratio", "income_per_member", "income_per_earner", "earner_ratio", "adm_year"]
rows = []
for c in num:
    a, o = A[c].dropna(), O[c].dropna()
    u = mannwhitneyu(a, o, method="asymptotic").statistic
    auc = u / (len(a) * len(o))
    lo, hi = o.quantile(.25), o.quantile(.75)
    rows.append({"feature": c, "AGROSOR median": a.median(), "JAGORON median": o.median(),
                 "AGROSOR p25-p75": f"{a.quantile(.25):,.0f}-{a.quantile(.75):,.0f}" if a.abs().max() > 20 else f"{a.quantile(.25):.2f}-{a.quantile(.75):.2f}",
                 "JAGORON p25-p75": f"{lo:,.0f}-{hi:,.0f}" if o.abs().max() > 20 else f"{lo:.2f}-{hi:.2f}",
                 "AUC": auc, "strength": abs(2 * auc - 1),
                 "AGROSOR inside JAGORON's middle 50%": ((a >= lo) & (a <= hi)).mean()})
N = pd.DataFrame(rows).sort_values("strength", ascending=False)
print("\n" + "\n1. NUMERIC FEATURES  (strength 0 = common to both, 1 = fully separated)\n")
print(N.round(3).to_string(index=False, line_width=2000))
N.round(4).to_csv(os.path.join(OUT, "agrosor_numeric.csv"), index=False)



1. NUMERIC FEATURES  (strength 0 = common to both, 1 = fully separated)

          feature  AGROSOR median  JAGORON median   AGROSOR p25-p75 JAGORON p25-p75  AUC  strength  AGROSOR inside JAGORON's middle 50%
   monthly_income       105000.00        45000.00    80,000-170,000   35,000-60,000 0.91      0.82                                 0.10
income_per_member        26000.00        12000.00     19,000-41,667    8,750-16,667 0.87      0.74                                 0.17
   yearly_savings       760000.00       270000.00 510,000-1,200,000 180,000-420,000 0.87      0.74                                 0.14
income_per_earner        65000.00        30000.00    40,000-110,000   20,000-40,000 0.82      0.64                                 0.26
  yearly_expenses       480000.00       240000.00   300,000-840,000 160,000-350,000 0.78      0.55                                 0.17
             land           30.00           10.00              0-99            0-45 0.61      0.23           

In [16]:
# If one person is randomly selected from the AGROSOR group and one person from anJAGORON group, what is the probability that the person from AGROSOR has a higher value? auc 50% mean equal

# AGROSOR inside JAGORON's middle 50% (mean on that 50% agrosor value behaviour less amount good for agrosor)
# strength if 0 then match and 1 is fully unmatch

In [37]:
#categorical features
def cramers_v(x, g, min_count=30):
    x = x.where(x.map(x.value_counts()) >= min_count, "(rare)")
    t = pd.crosstab(x, g)
    n = t.values.sum()
    chi2 = chi2_contingency(t, correction=False)[0]
    r, k = t.shape
    phi2 = max(0, chi2 / n - (k - 1) * (r - 1) / (n - 1))
    rc, kc = r - (r - 1) ** 2 / (n - 1), k - (k - 1) ** 2 / (n - 1)
    return float(np.sqrt(phi2 / max(1e-9, min(kc - 1, rc - 1)))), r


cats = ["purpose_family", "purpose", "district", "thana", "education", "occupation", "alt_occupation_group",
        "home_owner", "gender", "maritial_status"]
V = pd.DataFrame([{"feature": c, "Cramer's V": cramers_v(df[c].fillna("(missing)").astype(str), df["agrosor"])[0],
                   "categories": df[c].nunique()} for c in cats]).sort_values("Cramer's V", ascending=False)
print("\n" + "\n2. CATEGORICAL FEATURES  (Cramer's V: 0 = common to both, 1 = category decides the group)\n")
print(V.round(3).to_string(index=False)); V.round(4).to_csv(os.path.join(OUT, "agrosor_categorical_strength.csv"), index=False)


def cat_table(c, min_n=200, top=8, by="lift"):
    g = df.assign(v=df[c].fillna("(missing)").astype(str)).groupby("v").agg(loans=("agrosor", "size"), agrosor=("agrosor", "sum"))
    g["share of AGROSOR"] = g.agrosor / len(A); g["share of JAGORON"] = (g.loans - g.agrosor) / len(O)
    g["lift"] = g["share of AGROSOR"] / g["share of JAGORON"]; g["AGROSOR rate in category"] = g.agrosor / g.loans
    g = g[g.loans >= min_n]
    return g.sort_values("AGROSOR rate in category" if by == "rate" else "lift", ascending=False)


for c, mn in [("purpose_family", 100), ("occupation", 100), ("education", 300), ("alt_occupation_group", 100),
              ("home_owner", 100), ("gender", 100), ("maritial_status", 100)]:
    t = cat_table(c, mn, by="rate")
    print(f"\n--- {c}  (sorted by AGROSOR rate inside the category; overall rate {len(A) / len(df):.1%}) ---")
    print(t[["loans", "share of AGROSOR", "share of JAGORON", "lift", "AGROSOR rate in category"]].round(3).to_string())
    t.round(4).to_csv(os.path.join(OUT, f"agrosor_cat_{c}.csv"))
print("\n--- purpose (top 10 and bottom 5 by AGROSOR rate, categories with >= 300 loans) ---")
t = cat_table("purpose", 300, by="rate")
print(pd.concat([t.head(10), t.tail(5)])[["loans", "share of AGROSOR", "share of JAGORON", "lift", "AGROSOR rate in category"]].round(3).to_string())
t.round(4).to_csv(os.path.join(OUT, "agrosor_cat_purpose.csv"))
print("\n--- district (top 6 and bottom 4 by AGROSOR rate, >= 400 loans) ---")
t = cat_table("district", 400, by="rate")
print(pd.concat([t.head(6), t.tail(4)])[["loans", "share of AGROSOR", "lift", "AGROSOR rate in category"]].round(3).to_string())
# t.round(4).to_csv(os.path.join(OUT, "agrosor_cat_district.csv"))



2. CATEGORICAL FEATURES  (Cramer's V: 0 = common to both, 1 = category decides the group)

             feature  Cramer's V  categories
             purpose        0.41         242
      purpose_family        0.34           8
               thana        0.26         437
            district        0.23          64
              gender        0.17           2
           education        0.17          17
          occupation        0.14           6
alt_occupation_group        0.10           8
          home_owner        0.07           3
     maritial_status        0.03           5

--- purpose_family  (sorted by AGROSOR rate inside the category; overall rate 8.5%) ---
           loans  share of AGROSOR  share of JAGORON  lift  AGROSOR rate in category
v                                                                                   
Migration   5440              0.24              0.03  7.28                      0.46
Trade      20035              0.30              0.19  1.62          

In [41]:
### Posibility of aggrosor with this category
## lift high mean ( compare to JAGORON 9.15% better in this cats) agrosor is better in this category
##

In [39]:
# how the AGROSOR rate changes along the key numeric features
print("\n" +"\n3. AGROSOR RATE BY BAND (share of loans in the band that are AGROSOR)\n" )
bands = {"monthly_income": [-1, 30000, 40000, 50000, 60000, 70000, 80000, 100000, 150000, 1e12],
         "land": [-1, 0, 10, 30, 60, 100, 300, 1e5], "family": [-1, 2, 3, 4, 5, 6, 40], "earners": [-1, 1, 2, 3, 20],
         "age": [0, 25, 30, 35, 40, 45, 50, 55, 100], "expense_ratio": [-1, 0.1, 0.3, 0.5, 0.7, 1.0, 5]}
for c, b in bands.items():
    t = df.groupby(pd.cut(df[c], b), observed=True).agg(loans=("agrosor", "size"), AGROSOR_rate=("agrosor", "mean"))
    print(f"\n{c}"); print(t.round(3).T.to_string())

# two features together
print("\n" +  "\n4. AGROSOR RATE: monthly income band x purpose family  (loans in cell in brackets; cells < 100 loans blank)\n")
ib = pd.cut(df.monthly_income, [-1, 40000, 60000, 80000, 100000, 150000, 1e12], labels=["<=40k", "50-60k", "70-80k", "90-100k", "110-150k", ">150k"])
g = df.assign(ib=ib).groupby(["ib", "purpose_family"], observed=True).agg(n=("agrosor", "size"), r=("agrosor", "mean")).reset_index()
r = g.pivot(index="ib", columns="purpose_family", values="r").where(g.pivot(index="ib", columns="purpose_family", values="n") >= 100)
n = g.pivot(index="ib", columns="purpose_family", values="n")
print((r.round(2).astype(str) + " (" + n.fillna(0).astype(int).astype(str) + ")").where(r.notna(), "").to_string())




3. AGROSOR RATE BY BAND (share of loans in the band that are AGROSOR)


monthly_income
monthly_income  (-1.0, 30000.0]  (30000.0, 40000.0]  (40000.0, 50000.0]  (50000.0, 60000.0]  (60000.0, 70000.0]  (70000.0, 80000.0]  (80000.0, 100000.0]  (100000.0, 150000.0]  (150000.0, 1000000000000.0]
loans                  28424.00            25359.00            26040.00            13261.00             7996.00             6110.00              6251.00               4245.00                      6307.00
AGROSOR_rate               0.00                0.00                0.01                0.04                0.10                0.21                 0.32                  0.54                         0.47

land
land          (-1.0, 0.0]  (0.0, 10.0]  (10.0, 30.0]  (30.0, 60.0]  (60.0, 100.0]  (100.0, 300.0]  (300.0, 100000.0]
loans            51595.00     11765.00      19341.00      18018.00       12211.00         9669.00            1344.00
AGROSOR_rate         0.06         0.06          0.08       

In [40]:
# how the Jagoron rate changes along the key numeric features
print("\n" +"\n3. JAGORON RATE BY BAND (share of loans in the band that are JAGORON)\n" )
bands = {"monthly_income": [-1, 30000, 40000, 50000, 60000, 70000, 80000, 100000, 150000, 1e12],
         "land": [-1, 0, 10, 30, 60, 100, 300, 1e5], "family": [-1, 2, 3, 4, 5, 6, 40], "earners": [-1, 1, 2, 3, 20],
         "age": [0, 25, 30, 35, 40, 45, 50, 55, 100], "expense_ratio": [-1, 0.1, 0.3, 0.5, 0.7, 1.0, 5]}
for c, b in bands.items():
    t = df.groupby(pd.cut(df[c], b), observed=True).agg(loans=("jagoron", "size"), JAGORON_rate=("jagoron", "mean"))
    print(f"\n{c}"); print(t.round(3).T.to_string())

# two features together
print("\n" +  "\n4. JAGORON RATE: monthly income band x purpose family  (loans in cell in brackets; cells < 100 loans blank)\n")
ib = pd.cut(df.monthly_income, [-1, 40000, 60000, 80000, 100000, 150000, 1e12], labels=["<=40k", "50-60k", "70-80k", "90-100k", "110-150k", ">150k"])
g = df.assign(ib=ib).groupby(["ib", "purpose_family"], observed=True).agg(n=("jagoron", "size"), r=("jagoron", "mean")).reset_index()
r = g.pivot(index="ib", columns="purpose_family", values="r").where(g.pivot(index="ib", columns="purpose_family", values="n") >= 100)
n = g.pivot(index="ib", columns="purpose_family", values="n")
print((r.round(2).astype(str) + " (" + n.fillna(0).astype(int).astype(str) + ")").where(r.notna(), "").to_string())




3. JAGORON RATE BY BAND (share of loans in the band that are JAGORON)


monthly_income
monthly_income  (-1.0, 30000.0]  (30000.0, 40000.0]  (40000.0, 50000.0]  (50000.0, 60000.0]  (60000.0, 70000.0]  (70000.0, 80000.0]  (80000.0, 100000.0]  (100000.0, 150000.0]  (150000.0, 1000000000000.0]
loans                  28424.00            25359.00            26040.00            13261.00             7996.00             6110.00              6251.00               4245.00                      6307.00
JAGORON_rate               0.76                0.79                0.79                0.79                0.74                0.66                 0.57                  0.38                         0.42

land
land          (-1.0, 0.0]  (0.0, 10.0]  (10.0, 30.0]  (30.0, 60.0]  (60.0, 100.0]  (100.0, 300.0]  (300.0, 100000.0]
loans            51595.00     11765.00      19341.00      18018.00       12211.00         9669.00            1344.00
JAGORON_rate         0.77         0.77          0.74       

In [8]:
class FeatureBuilder:
    def __init__(self, te_smooth=20, n_splits=5, seed=0):
        self.m, self.n_splits, self.seed = te_smooth, n_splits, seed

    # stage 1: row-wise features
    def _rows(self, df):
        inc = df["total_monthly_income"].clip(upper=self.inc_cap)
        yinc = df["yearly_total_income"].replace(0, np.nan)
        expn = df["yearly_total_expenses"].clip(upper=self.exp_cap)
        sav = df["yearly_total_savings"].clip(lower=-self.exp_cap, upper=self.exp_cap)
        fam = df["total_family_member"].clip(upper=12)
        fam_nz = fam.replace(0, np.nan)
        earn = df["total_earning_member"].clip(upper=8)
        earn_nz = earn.replace(0, np.nan)
        land = df["farm_land_amt"].where(df["farm_land_amt"] < 1e5)
        adm = pd.to_datetime(df["admission_date"])
        age = ((adm - pd.to_datetime(df["dateofbirth"])).dt.days / 365.25).clip(18, 70)
        pur = clean_text_purpose(df["loan_use_sources"])
        pur_c = pur.where(pur.isin(self.purposes), "rare")
        pg = pur.map(purpose_group)
        X = pd.DataFrame(index=df.index)
        #cleaned raw numeric
        X["monthly_income"] = inc
        X["land"] = land
        X["family_size"] = fam
        X["earners"] = earn
        X["yearly_expenses"] = expn
        X["yearly_savings"] = sav
        X["rent_years"] = df["year_of_rent"].clip(upper=60)
        X["age"] = age
        X["edu_ord"] = df["education"].map(EDU_MAP)
        X["adm_ord"] = (adm - pd.Timestamp("2017-01-01")).dt.days
        #engineered numeric
        X["exp_ratio"] = (df["yearly_total_expenses"] / yinc).clip(0, 3)
        X["sav_rate"] = (df["yearly_total_savings"] / yinc).clip(-3, 3)
        X["sav_mismatch"] = (df["yearly_total_savings"] != df["yearly_total_income"] - df["yearly_total_expenses"]).astype(int)
        X["inc_suspect"] = (X["exp_ratio"] < 0.10).astype(int)       # expenses < 10% of income: income likely overstated
        X["inc_per_member"] = inc / fam_nz
        X["inc_per_earner"] = inc / earn_nz
        X["earner_ratio"] = earn / fam_nz
        X["dependents"] = (fam - earn).clip(lower=0)
        X["dep_ratio"] = X["dependents"] / earn_nz
        X["land_zero"] = (df["farm_land_amt"] == 0).astype(int)
        X["land_per_member"] = land / fam_nz
        X["inc_round10k"] = (df["total_monthly_income"] % 10000 == 0).astype(int)
        X["inc_round50k"] = (df["total_monthly_income"] % 50000 == 0).astype(int)
        X["alt_flag"] = df["alternative_occupation"].notna().astype(int)
        X["home_missing"] = df["home_owner_type"].isna().astype(int)
        X["purpose_n"] = np.log1p(pur.map(self.purpose_counts).fillna(0))
        #categoricals
        def cat(s, cats): return pd.Categorical(s.where(s.isin(cats)), categories=cats)
        X["purpose"] = cat(pur_c, self.purposes + ["rare"])
        X["pgroup"] = cat(pg, self.pgroups)
        X["district"] = cat(df["current_addr_district"].astype(str).str.strip(), self.districts)
        X["thana"] = cat(df["current_addr_thana"].astype(str).str.strip(), self.thanas)
        X["occupation"] = cat(df["occupation"].fillna("NA").astype(str), self.occs)
        X["altocc"] = cat(df["alternative_occupation"].map(altocc_group), self.altoccs)
        X["home"] = cat(df["home_owner_type"].fillna("NA").astype(str), ["Own", "Rent", "NA"])
        X["marital"] = cat(df["maritial_status"].astype(str), self.maritals)
        X["female"] = (df["gender"] == "Female").astype(int)
        if "loan_Component" in df:
            c = df["loan_Component"].astype(str)
            c = np.where(c.str.startswith("AGROSOR"), "AGROSOR", np.where(c.isin(["JAGORON", "SUFOLON"]), c, "other"))
            X["comp"] = pd.Categorical(c, categories=["JAGORON", "SUFOLON", "AGROSOR", "other"])

        incband = pd.cut(inc, [-1, 30000, 40000, 50000, 60000, 70000, 80000, 100000, 1e12]).astype(str)
        K = pd.DataFrame({"prop": clean_id(df["proposer_oldid"]), "thana": df["current_addr_thana"].astype(str).str.strip(),
                          "district": df["current_addr_district"].astype(str).str.strip(), "purpose": pur_c.astype(str),
                          "altocc": X["altocc"].astype(str), "pur_inc": pur_c.astype(str) + "|" + incband,
                          "thana_pg": df["current_addr_thana"].astype(str).str.strip() + "|" + pg}, index=df.index)
        return X, K

    # peer statistics learned on training rows
    """
    Person Income high or low compared to same loan_amount, based on geographical area income high or low, common on property
    type, % on rented house, landownership differ from local avg
    """
    def _fit_peers(self, X, K):   #original numerical and categorical with grouping
        g = lambda key, col, fn: X[col].groupby(K[key].values).agg(fn)
        self.peer = {
            "pur_inc_med": g("purpose", "monthly_income", "median"),
            "thana_inc_med": g("thana", "monthly_income", "median"),
            "thana_land": g("thana", "land", "mean"),
            "thana_rent": X["home"].eq("Rent").groupby(K["thana"].values).mean(),
            "thana_n": K["thana"].value_counts(),
            "dist_inc_med": g("district", "monthly_income", "median"),
            "dist_home_missing": g("district", "home_missing", "mean"),
            "prop_n": K["prop"].value_counts(),
            "prop_inc_med": g("prop", "monthly_income", "median"),
        }
        self.glob = {"inc": float(X["monthly_income"].median()), "land": float(X["land"].mean())}

    def _apply_peers(self, X, K):
        P, G = self.peer, self.glob
        X["pur_inc_med"] = K["purpose"].map(P["pur_inc_med"]).fillna(G["inc"]).values
        X["thana_inc_med"] = K["thana"].map(P["thana_inc_med"]).fillna(G["inc"]).values
        X["thana_land"] = K["thana"].map(P["thana_land"]).fillna(G["land"]).values
        X["thana_rent"] = K["thana"].map(P["thana_rent"]).fillna(0).values
        X["thana_n"] = np.log1p(K["thana"].map(P["thana_n"]).fillna(0).values)
        X["dist_inc_med"] = K["district"].map(P["dist_inc_med"]).fillna(G["inc"]).values
        X["dist_home_missing"] = K["district"].map(P["dist_home_missing"]).fillna(0).values   # proxy for branch data-capture habit
        X["prop_n"] = np.log1p(K["prop"].map(P["prop_n"]).fillna(0).values)
        X["prop_inc_med"] = K["prop"].map(P["prop_inc_med"]).fillna(G["inc"]).values   # should used it or not, single proposer loanee can make issue
        X["inc_rel_purpose"] = (X["monthly_income"] / X["pur_inc_med"]).clip(0, 10)         # income vs median income with same purpose
        X["inc_rel_thana"] = (X["monthly_income"] / X["thana_inc_med"]).clip(0, 10)
        X["inc_rel_prop"] = (X["monthly_income"] / X["prop_inc_med"]).clip(0, 10)
        X["land_rel_thana"] = X["land"] / X["thana_land"].replace(0, np.nan)
        return X

    #target encodings
    TE_SPECS = {"te_prop": ("prop", "log"), "te_thana": ("thana", "log"), "te_district": ("district", "log"),
                "te_purpose": ("purpose", "log"), "te_altocc": ("altocc", "log"), "te_pur_inc": ("pur_inc", "log"),
                "te_thana_pg": ("thana_pg", "log"), "tb_prop": ("prop", "big"), "tb_thana": ("thana", "big"),
                "tb_pur_inc": ("pur_inc", "big")}

    def _te_table(self, key, target):     # smoothing the encoding
        return pd.DataFrame({"s": target, "k": key}).groupby("k")["s"].agg(["sum", "count"])

    def _te_apply(self, key, tab, prior):
        s = key.map(tab["sum"]).fillna(0).values   # sum of target value of  category
        c = key.map(tab["count"]).fillna(0).values  # number of observation on k category
        return (s + prior * self.m) / (c + self.m)   # p is global target mean and m is smoothing(how much trust the global avg)

    def _targets(self, y):
        return {"log": np.log1p(np.asarray(y, float)), "big": (np.asarray(y, float) >= BIG).astype(float)}

    #public API
    def fit_transform(self, df, y):
        d = df
        self.inc_cap = float(d["total_monthly_income"].quantile(0.999))
        self.exp_cap = float(d["yearly_total_expenses"].quantile(0.999))
        pur = clean_text_purpose(d["loan_use_sources"])
        vc = pur.value_counts()
        self.purposes = sorted(vc[vc >= 100].index.tolist())
        self.purpose_counts = vc
        self.pgroups = sorted(pur.map(purpose_group).unique().tolist())
        self.districts = sorted(d["current_addr_district"].astype(str).str.strip().unique().tolist())
        self.thanas = sorted(d["current_addr_thana"].astype(str).str.strip().unique().tolist())
        self.occs = sorted(d["occupation"].fillna("NA").astype(str).unique().tolist())
        self.altoccs = sorted(d["alternative_occupation"].map(altocc_group).unique().tolist())
        self.maritals = sorted(d["maritial_status"].astype(str).unique().tolist())
        X, K = self._rows(d)
        self._fit_peers(X, K)
        X = self._apply_peers(X, K)
        T = self._targets(y)
        self.prior = {k: float(v.mean()) for k, v in T.items()}
        self.te_tables = {name: self._te_table(K[key], T[kind]) for name, (key, kind) in self.TE_SPECS.items()}   #smoothing
        kf = KFold(self.n_splits, shuffle=True, random_state=self.seed)  # time based dataset  shuffle False, out-of-fold encodings for training rows
        for name, (key, kind) in self.TE_SPECS.items():
            out = np.zeros(len(X))
            for a, b in kf.split(X):
                tab = self._te_table(
                    K[key].iloc[a],
                    T[kind][a])

                out[b] = self._te_apply(
                    K[key].iloc[b],
                    tab,
                    self.prior[kind])
            X[name] = out
        return X


    def transform(self, df):
        X, K = self._rows(df)
        X = self._apply_peers(X, K)
        for name, (key, kind) in self.TE_SPECS.items():
            X[name] = self._te_apply(K[key], self.te_tables[name], self.prior[kind])
        return X

In [9]:
df = load()
y = df["loan_amount"].values
n = len(df)
cut = int(n * 0.8)
fb = FeatureBuilder()
Xtr = fb.fit_transform(df.iloc[:cut], y[:cut])
Xte = fb.transform(df.iloc[cut:])
te = df.iloc[cut:].reset_index(drop=True)
Xte = Xte.reset_index(drop=True)
ytr = np.log1p(y[:cut])
a = y[cut:]
k = int(cut * 0.85)
big = a >= 300000

In [12]:
# P(big): chance a member with this PROFILE (no component, no amount) gets a >=300k loan
p = dict(PARAMS, objective="binary", random_state=0, n_estimators=2500)
p.update(BEST)
clf = lgb.LGBMClassifier(**p).fit(Xtr.iloc[:k][SA], (y[:cut] >= 300000)[:k],
                                  eval_set=[(Xtr.iloc[k:][SA], (y[:cut] >= 300000)[k:])],
                                  callbacks=[lgb.early_stopping(50, verbose=False)])
pb = clf.predict_proba(Xte[SA])[:, 1]


def quantile_pred(alpha):
    q = dict(PARAMS, objective="quantile", alpha=alpha, random_state=0, n_estimators=2500); q.update(BEST)
    m = lgb.LGBMRegressor(**q).fit(Xtr.iloc[:k][SA], ytr[:k], eval_set=[(Xtr.iloc[k:][SA], ytr[k:])],
                                   eval_metric="quantile", callbacks=[lgb.early_stopping(50, verbose=False)])
    return np.clip(np.expm1(m.predict(Xte[SA])), 4000, 5e6)


Q = {q: quantile_pred(q) for q in [0.1, 0.5, 0.9, 0.95]}
comp = te["loan_Component"].astype(str).to_numpy(dtype=str)
is_agrosor = np.char.startswith(comp, "AGROSOR")
is_jagoron = np.char.startswith(comp, "JAGORON")
print(f"test loans {len(a):,} | loans >= 300k: {big.sum():,} | component AGROSOR*: {is_agrosor.sum():,} "
      f"| overlap {np.sum(big & is_agrosor):,}\nclassifier AUC {roc_auc_score(big, pb):.3f}", flush=True)

yearly = te["total_monthly_income"].clip(lower=5000).values * 12
breach = a / yearly > np.where(is_agrosor, 2 / 3, 1 / 3)
flag = Xte["inc_suspect"].values == 1          # expenses < 10% of income


def wilson(x, m, z=1.96):
    if m == 0:
        return (np.nan, np.nan)
    ph = x / m
    d = 1 + z * z / m
    c = (ph + z * z / (2 * m)) / d
    h = z * np.sqrt(ph * (1 - ph) / m + z * z / (4 * m * m)) / d
    return c - h, c + h


test loans 24,799 | loans >= 300k: 2,112 | component AGROSOR*: 2,229 | overlap 2,112
classifier AUC 0.961


In [14]:
bins = pd.cut(pb[big], [0, .1, .2, .35, .5, .65, .8, 1.0], include_lowest=True)
t = pd.Series(bins).value_counts(sort=False).to_frame("loans"); t["share"] = t.loans / big.sum()
print("big loans (>= 300k) whose profile gives P(big) < 0.2")
print(f"\nDistribution of P(big) among the {big.sum():,} loans that really were >= 300k:\n{t.round(3).to_string()}")
low = big & (pb < 0.2)
rest = big & (pb >= 0.2)
lo_lo, lo_hi = wilson(low.sum(), big.sum())
print(f"\nLOW group (P<0.2): {low.sum()} of {big.sum()} = {low.sum() / big.sum():.1%}  (95% CI {lo_lo:.1%}-{lo_hi:.1%})"
      f"  | median P(big) {np.median(pb[low]):.3f}")
alt_all = is_agrosor & (pb < 0.2)
print(f"Same check using the COMPONENT label instead of amount: AGROSOR* loans with P<0.2: {alt_all.sum()} of {is_agrosor.sum()} = {alt_all.sum() / is_agrosor.sum():.1%}")

inc = te["total_monthly_income"].values
def prof(mask):
    d = {"n": int(mask.sum()), "median monthly income": np.median(inc[mask]),
         "income <= 40k": np.mean(inc[mask] <= 40000), "income >= 100k": np.mean(inc[mask] >= 100000),
         "median land": np.nanmedian(Xte["land"].values[mask]), "land > 100": np.nanmean(Xte["land"].values[mask] > 100),
         "mean family size": Xte["family_size"].values[mask].mean(), "mean education (0-9)": np.nanmean(Xte["edu_ord"].values[mask]),
         "Migration purpose share": np.mean((Xte["pgroup"].astype(str) == "Migration").values[mask]),
         "renter share": np.mean((Xte["home"].astype(str) == "Rent").values[mask]),
         "income vs same-purpose peers (x)": np.median(Xte["inc_rel_purpose"].values[mask]),
         "past big-loan rate, same purpose x income cell": np.mean(Xte["tb_pur_inc"].values[mask]),
         "past big-loan rate, same proposer": np.mean(Xte["tb_prop"].values[mask]),
         "median requested amount": np.median(a[mask]), "exactly 300,000": np.mean(a[mask] == 300000), ">= 500,000": np.mean(a[mask] >= 500000),
         "breaches income cap (2/3 yearly)": np.mean(breach[mask]), "income-overstated flag": np.mean(flag[mask])}
    return d
P1 = pd.DataFrame({"LOW (P<0.2)": prof(low), "JAGORON big loans": prof(rest), "ALL test loans": prof(np.ones(len(a), bool))})
print("\nWho are they?  (profile comparison)")
print(P1.round(3).to_string())

# why: where do they sit?
print("\nTop purposes in the LOW group (with how often that purpose gets a >=300k loan in the test period):")
pur = Xte["purpose"].astype(str)
tp = pd.DataFrame({"purpose": pur[low].value_counts().head(8)})
tp.columns = ["n_low"]
tp["share_of_low"] = tp.n_low / low.sum(); tp["big_rate_of_purpose"] = [np.mean(big[(pur == i).values]) for i in tp.index]
print(tp.round(3).to_string())
dist = te["current_addr_district"].astype(str).str.strip()
dd = pd.DataFrame({"n_low": dist[low].value_counts(), "n_all": dist.value_counts()}).dropna().sort_values("n_low", ascending=False).head(6)
dd["share_of_low"] = dd.n_low / low.sum(); dd["share_of_all"] = dd.n_all / len(a); dd["lift"] = dd.share_of_low / dd.share_of_all
print("\nTop districts in the LOW group:")
print(dd.round(3).to_string())
prop = te["proposer_oldid"].astype("string").str.strip().str.lstrip("0").fillna("NA")
pc = prop[low].value_counts(); print(f"\nProposers: {pc.shape[0]} distinct among {low.sum()} low-P big loans; top-5 proposers cover {pc.head(5).sum() / low.sum():.1%}; "
                                    f"proposers with >=3 such loans: {(pc >= 3).sum()}")
print("\nIncome bands of the LOW group vs JAGORON big loans (share):")
ib = pd.cut(inc, [-1, 30000, 40000, 50000, 70000, 100000, 1e12], labels=["<=30k", "40k", "50k", "60-70k", "80-100k", ">100k"])
print(pd.DataFrame({"LOW": pd.Series(ib[low]).value_counts(normalize=True, sort=False), "JAGORON big": pd.Series(ib[rest]).value_counts(normalize=True, sort=False)}).round(3).to_string())
out1 = te.loc[low, ["member_id", "total_monthly_income", "loan_use_sources", "current_addr_district", "current_addr_thana", "farm_land_amt", "total_family_member", "loan_Component", "loan_amount"]].copy()
out1["P_big"] = pb[low].round(3); out1["predicted_median"] = Q[0.5][low].round(0); out1["predicted_P90"] = Q[0.9][low].round(0)
out1.sort_values("P_big").to_csv(os.path.join(R, "claim1_big_loans_low_pbig.csv"), index=False)

big loans (>= 300k) whose profile gives P(big) < 0.2

Distribution of P(big) among the 2,112 loans that really were >= 300k:
               loans  share
(-0.001, 0.1]    212   0.10
(0.1, 0.2]       173   0.08
(0.2, 0.35]      228   0.11
(0.35, 0.5]      234   0.11
(0.5, 0.65]      276   0.13
(0.65, 0.8]      315   0.15
(0.8, 1.0]       674   0.32

LOW group (P<0.2): 385 of 2112 = 18.2%  (95% CI 16.6%-19.9%)  | median P(big) 0.087
Same check using the COMPONENT label instead of amount: AGROSOR* loans with P<0.2: 477 of 2229 = 21.4%

Who are they?  (profile comparison)
                                                LOW (P<0.2)  JAGORON big loans  ALL test loans
n                                                    385.00            1727.00        24799.00
median monthly income                              70000.00          130000.00        50000.00
income <= 40k                                          0.05               0.00            0.44
income >= 100k                                

In [15]:
print("\n" + "\nCLAIM 2: loans BELOW 300k whose profile gives P(big) > 0.5\n")
nb = ~big & is_jagoron
fp = nb & (pb > 0.5)
flagged = pb > 0.5
print(f"non-big loans: {nb.sum():,} | with P>0.5: {fp.sum()} = {fp.sum() / nb.sum():.2%} | all loans with P>0.5: {flagged.sum()} "
      f"of which truly >=300k: {np.sum(flagged & big)} (precision {np.mean(big[flagged]):.1%}, recall {np.mean(flagged[big]):.1%})")
band = pd.cut(a, [0, 100000, 150000, 200000, 250000, 290000, 299999], labels=["<=100k", "120-150k", "200k", "250k", "260-290k", "295-299k"])
cmp = pd.DataFrame({"P>0.5 (n=%d)" % fp.sum(): pd.Series(band[fp]).value_counts(normalize=True, sort=False),
                    "P<=0.5 (n=%d)" % (nb & ~flagged).sum(): pd.Series(band[nb & ~flagged]).value_counts(normalize=True, sort=False)})
print("\nHow large were the loans they actually took?  (share)")
print(cmp.round(3).to_string())
print(f"\nmedian actual {np.median(a[fp]):,.0f} | median predicted-median {np.median(Q[0.5][fp]):,.0f} | actual >= 200k: {np.mean(a[fp] >= 200000):.1%} "
      f"| actual >= 150k: {np.mean(a[fp] >= 150000):.1%}")
print("Most common amounts:", a[fp].astype(int).tolist() and pd.Series(a[fp].astype(int)).value_counts().head(6).to_dict())
print("Component mix:", pd.Series(comp[fp]).value_counts(normalize=True).round(3).head(5).to_dict())
P2 = pd.DataFrame({"P>0.5 but < 300k": prof(fp), "P<=0.5 and < 300k": prof(nb & ~flagged), "truly big & P>0.5": prof(big & flagged)})
print("\nProfile comparison")
print(P2.round(3).to_string())
lo2, hi2 = wilson(fp.sum(), nb.sum())
print(f"\n95% CI of the 2.2% share: {lo2:.2%}-{hi2:.2%}")
out2 = te.loc[fp, ["member_id", "total_monthly_income", "loan_use_sources", "current_addr_district", "farm_land_amt", "loan_Component", "loan_amount"]].copy()
out2["P_big"] = pb[fp].round(3); out2["predicted_median"] = Q[0.5][fp].round(0)
out2.sort_values("P_big", ascending=False).to_csv(os.path.join(R, "claim2_nonbig_loans_high_pbig.csv"), index=False)




CLAIM 2: loans BELOW 300k whose profile gives P(big) > 0.5

non-big loans: 18,020 | with P>0.5: 414 = 2.30% | all loans with P>0.5: 1770 of which truly >=300k: 1265 (precision 71.5%, recall 59.9%)

How large were the loans they actually took?  (share)
          P>0.5 (n=414)  P<=0.5 (n=17606)
<=100k             0.30              0.88
120-150k           0.14              0.05
200k               0.53              0.07
250k               0.03              0.00
260-290k           0.00              0.00
295-299k           0.00              0.00

median actual 200,000 | median predicted-median 274,209 | actual >= 200k: 55.8% | actual >= 150k: 68.8%
Most common amounts: {200000: 218, 100000: 58, 150000: 54, 80000: 24, 90000: 16, 250000: 13}
Component mix: {'JAGORON': 1.0}

Profile comparison
                                                P>0.5 but < 300k  P<=0.5 and < 300k  truly big & P>0.5
n                                                         414.00           17606.00            1265

In [39]:
print("\n" + "\nCLAIM 3: is the income-overstated flag more common among stretched requests?\n")
print("flag = yearly expenses < 10% of yearly income")
zone = np.where(a > Q[0.95], "4 red: above P95", np.where(a > Q[0.9], "3 amber: P90-P95", np.where(a < Q[0.1], "1 below P10", "2 inside P10-P90")))
rows = []
for zname in sorted(set(zone)):
    m = zone == zname
    lo_, hi_ = wilson(flag[m].sum(), m.sum())
    rows.append([zname, int(m.sum()), int(flag[m].sum()), flag[m].mean(), lo_, hi_])
Z = pd.DataFrame(rows, columns=["zone", "loans", "flagged", "flag share", "CI low", "CI high"])
print(Z.round(3).to_string(index=False))
tot = flag.mean(); print(f"overall flag share {tot:.3f}")
chi = chi2_contingency(pd.crosstab(zone, flag))
print(f"chi-square across the 4 zones: p = {chi[1]:.4f}")
red, ex = zone == "4 red: above P95", zone != "4 red: above P95"
ab, nab = a > Q[0.9], a <= Q[0.9]
def rr(mask_a, mask_b):
    a1, n1, b1, n2 = flag[mask_a].sum(), mask_a.sum(), flag[mask_b].sum(), mask_b.sum()
    r = (a1 / n1) / (b1 / n2); se = np.sqrt(1 / a1 - 1 / n1 + 1 / b1 - 1 / n2); return r, r * np.exp(-1.96 * se), r * np.exp(1.96 * se)
r_ = rr(red, ex); print(f"red vs everything else: flag share {flag[red].mean():.3f} vs {flag[ex].mean():.3f} -> relative risk {r_[0]:.2f} (95% CI {r_[1]:.2f}-{r_[2]:.2f})")
r_ = rr(ab, nab); print(f"above P90 vs inside/below: relative risk {r_[0]:.2f} (95% CI {r_[1]:.2f}-{r_[2]:.2f})")
bl = zone == "1 below P10"
r_ = rr(bl, ~bl)
print(f"BELOW P10 vs rest: flag share {flag[bl].mean():.3f} vs {flag[~bl].mean():.3f} -> relative risk {r_[0]:.2f} (95% CI {r_[1]:.2f}-{r_[2]:.2f})")
stretch = a / Q[0.5]
print(f"\nmedian stretch (requested / predicted median): flagged {np.median(stretch[flag]):.2f} vs not flagged {np.median(stretch[~flag]):.2f}")
print(f"median loan / monthly income:                   flagged {np.median(a[flag] / inc[flag].clip(min=5000)):.2f} vs not flagged {np.median(a[~flag] / inc[~flag].clip(min=5000)):.2f}")
print("\nRobustness - does the answer change with the flag threshold?  (share of flagged loans that are above P90 / above P95)")
er = (te["yearly_total_expenses"] / te["yearly_total_income"].replace(0, np.nan)).values
for thr in [0.05, 0.10, 0.15, 0.20, 0.25]:
    f = er < thr; print(f"   expenses < {thr:.0%} of income: flagged {f.mean():5.1%} of loans | above P90: {np.mean(a[f] > Q[0.9][f]):.1%} (rest {np.mean(a[~f] > Q[0.9][~f]):.1%}) | above P95: {np.mean(a[f] > Q[0.95][f]):.1%} (rest {np.mean(a[~f] > Q[0.95][~f]):.1%})")
print("\nJAGORON data-quality signals by zone (share of loans):")
JAGORON = pd.DataFrame({"savings != income - expenses": Xte["sav_mismatch"].values == 1, "income is multiple of 50k": Xte["inc_round50k"].values == 1,
                      "income >= 200k/month": inc >= 200000, "land entry error (>=100k units)": te["farm_land_amt"].values >= 1e5,
                      "family size 0 or >= 12": (te["total_family_member"].values == 0) | (te["total_family_member"].values >= 12),
                      # signs that income may be UNDERSTATED (the direction that would make a request look like a stretch)
                      "expenses >= income": (te["yearly_total_expenses"].values >= te["yearly_total_income"].values),
                      "negative savings": te["yearly_total_savings"].values < 0,
                      "zone": zone})
print(JAGORON.groupby("zone").mean().round(3).to_string())

print("\n Why the direction matters: the model uses income as an input.")
print("  income OVERSTATED  -> predicted capacity too high -> the request looks SMALL  (lands in 'below P10')")
print("  income UNDERSTATED -> predicted capacity too low  -> the request looks LARGE  (lands in 'amber/red')")
print("so the overstated-income flag can only explain low-zone cases; the 'expenses >= income' / 'negative savings' rows above are the relevant check for the red zone.")



CLAIM 3: is the income-overstated flag more common among stretched requests?

flag = yearly expenses < 10% of yearly income
            zone  loans  flagged  flag share  CI low  CI high
     1 below P10   2759      376        0.14    0.12     0.15
2 inside P10-P90  19123     2354        0.12    0.12     0.13
3 amber: P90-P95   1447      173        0.12    0.10     0.14
4 red: above P95   1470      152        0.10    0.09     0.12
overall flag share 0.123
chi-square across the 4 zones: p = 0.0196
red vs everything else: flag share 0.103 vs 0.124 -> relative risk 0.83 (95% CI 0.71-0.97)
above P90 vs inside/below: relative risk 0.89 (95% CI 0.80-0.99)
BELOW P10 vs rest: flag share 0.136 vs 0.122 -> relative risk 1.12 (95% CI 1.01-1.24)

median stretch (requested / predicted median): flagged 1.01 vs not flagged 1.01
median loan / monthly income:                   flagged 1.33 vs not flagged 1.60

Robustness - does the answer change with the flag threshold?  (share of flagged loans that a

In [ ]:
# Confidence Interval
# 15705 on the whole dataset